In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.weightstats import ztest
rng = np.random.default_rng(42)


In [2]:
titanic = pd.read_csv("train_data.csv")

In [3]:
titanic.head()


,Unnamed: 0,PassengerId,Survived,Sex,Age,Fare,Pclass_1,Pclass_2,Pclass_3,Family_size,Title_1,Title_2,Title_3,Title_4,Emb_1,Emb_2,Emb_3
0,0,1,0,1,0.2750,0.014151,0,0,1,0.1,1,0,0,0,0,0,1
1,1,2,1,0,0.4750,0.139136,1,0,0,0.1,1,0,0,0,1,0,0
2,2,3,1,0,0.3250,0.015469,0,0,1,0.0,0,0,0,1,0,0,1
3,3,4,1,0,0.4375,0.103644,1,0,0,0.1,1,0,0,0,0,0,1
4,4,5,0,1,0.4375,0.015713,0,0,1,0.0,1,0,0,0,0,0,1


In [4]:
first_class = titanic.loc[titanic["Pclass_1"].eq(1)].copy()
lower_classes = titanic.loc[titanic[["Pclass_2", "Pclass_3"]].eq(1).any(axis=1)].copy()


In [5]:
first_class.head()


,Unnamed: 0,PassengerId,Survived,Sex,Age,Fare,Pclass_1,Pclass_2,Pclass_3,Family_size,Title_1,Title_2,Title_3,Title_4,Emb_1,Emb_2,Emb_3
1,1,2,1,0,0.4750,0.139136,1,0,0,0.1,1,0,0,0,1,0,0
3,3,4,1,0,0.4375,0.103644,1,0,0,0.1,1,0,0,0,0,0,1
6,6,7,0,1,0.6750,0.101229,1,0,0,0.0,1,0,0,0,0,0,1
11,11,12,1,0,0.7250,0.051822,1,0,0,0.0,0,0,0,1,0,0,1
23,23,24,1,1,0.3500,0.069291,1,0,0,0.0,1,0,0,0,0,0,1


In [6]:
first_class_survival = first_class["Survived"].mean()
lower_classes_survival = lower_classes["Survived"].mean()

first_class_survival, lower_classes_survival


(np.float64(0.616580310880829), np.float64(0.3121869782971619))

In [7]:
bootstrap_differences = []

for _ in range(10_000):
    first_class_sample = rng.choice(
        first_class["Survived"].to_numpy(),
        size=len(first_class),
        replace=True,
    )
    lower_classes_sample = rng.choice(
        lower_classes["Survived"].to_numpy(),
        size=len(lower_classes),
        replace=True,
    )
    bootstrap_differences.append(
        first_class_sample.mean() - lower_classes_sample.mean()
    )


In [8]:
np.quantile(bootstrap_differences, q=[0.025, 0.975])

array([0.22569286, 0.38142587])

In [9]:
overall_survival = titanic["Survived"].mean()
observed_difference = first_class_survival - lower_classes_survival
standard_error = np.sqrt(overall_survival * (1 - overall_survival) / len(titanic["Survived"]))
observed_difference / standard_error


np.float64(17.59316840863016)

In [10]:
ztest(first_class["Survived"], lower_classes["Survived"])

(np.float64(7.830586337563221), np.float64(4.855999313122102e-15))